In [1]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
# KGAT_2ec3cbfb8183b69ded54729368abbf53
kagglehub.login()


In [3]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

piotereks_glovec_pl_100d_path = kagglehub.dataset_download('piotereks/glovec-pl-100d')

print('Data source import complete.')


100%|██████████| 715M/715M [00:50<00:00, 14.8MB/s] 

Extracting files...


Data source import complete.


In [ ]:
!pip install fasttext faiss-cpu gensim
!apt-get install -y p7zip-full

In [ ]:
!pip install fasttext faiss-cpu gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.1 MB/s eta 0:00:00:00:0100:01


In [13]:
all_words.index("wpisz")

ValueError: 'wpisz' is not in list

In [ ]:
import os
import numpy as np
import faiss
from gensim.models import KeyedVectors

# Simple loader: focused behavior for .npy vector files
# Set these paths before running the cell
main_vec_file = 'datasets/fasttext_100_3_polish.bin.wv.vectors.npy'
main_vocab_file = None  # set to a .txt or .npy file containing words (one per line)

def _load_simple_npy_vectors(vec_path, vocab_path=None):
    vecs = np.load(vec_path, allow_pickle=True)
    if vecs.ndim != 2 or not np.issubdtype(vecs.dtype, np.floating):
        raise ValueError(f"Expected 2D float array in {vec_path}, got shape={vecs.shape}, dtype={vecs.dtype}")
    words = None
    # If user provided a vocab path, load it
    if vocab_path:
        if vocab_path.endswith('.npy'):
            words = [str(x) for x in np.load(vocab_path, allow_pickle=True).tolist()]
        else:
            with open(vocab_path, 'r', encoding='utf-8') as f:
                words = [ln.strip() for ln in f if ln.strip()]
    else:
        # Try a few common sibling filenames near the vectors file
        base = vec_path
        for suffix in ('.wv.vectors_vocab.npy', '.wv.vectors_vocab.txt', '.words.npy', '.words.txt', '.vocab.npy', '.vocab.txt'):
            cand = base.replace('.wv.vectors.npy', '') + suffix if '.wv.vectors.npy' in base else base[:-4] + suffix
            if os.path.exists(cand):
                if cand.endswith('.npy'):
                    words = [str(x) for x in np.load(cand, allow_pickle=True).tolist()]
                else:
                    with open(cand, 'r', encoding='utf-8') as f:
                        words = [ln.strip() for ln in f if ln.strip()]
                break
    if words is None:
        raise ValueError('Loaded .npy vectors but could not find a matching vocab file. Set `main_vocab_file` to a vocab .txt or .npy file.')
    if len(words) != vecs.shape[0]:
        raise ValueError(f'Vocab length ({len(words)}) does not match vectors rows ({vecs.shape[0]})')
    kv = KeyedVectors(vector_size=vecs.shape[1])
    kv.add_vectors(words, vecs.astype('float32'))
    return kv

# --- 1. Load embeddings (simple .npy support) ---
ext = os.path.splitext(main_vec_file)[1].lower()
if ext == '.npy':
    wv = _load_simple_npy_vectors(main_vec_file, vocab_path=main_vocab_file)
else:
    raise ValueError('This simple loader is configured only for .npy vector files. Set `main_vec_file` to a .npy file.')

# --- 2. Prepare matrix ---
all_words = list(wv.key_to_index.keys())
all_vectors = np.array([wv[w] for w in all_words]).astype('float32')

# --- 3. Normalize once (IMPORTANT) ---
faiss.normalize_L2(all_vectors)

# --- 4. Build FAISS index (exact search) ---
d = all_vectors.shape[1]
index = faiss.IndexFlatIP(d)
index.add(all_vectors)


In [21]:
v_D_candidate = v_C + X_diff
scores, indices = index.search(v_D_candidate.reshape(1, -1), k)
print(scores)
print(indices)
faiss.normalize_L2(v_D_candidate.reshape(1, -1))
scores, indices = index.search(v_D_candidate.reshape(1, -1), k)
print(scores)
print(indices)


[[5.2400303 4.3025036 4.226172  4.1048956 4.057139  4.0301194 3.9640222
  3.933627  3.9007957 3.8252735 3.81199   3.795555  3.792898  3.7311802
  3.731007  3.7267718 3.7220678 3.6956656 3.6602287 3.6533055 3.648355
  3.641589  3.6389983 3.6323147 3.6166053]]
[[  8338  25606  16474  14872  29538  10626   6612  47205 308334    194
    6517   6482  10760  49232   3940  41513   5740  29210  16362   6009
   15533  21267  12044   4871  14973]]
[[0.7316457  0.60074246 0.5900845  0.5731511  0.5664829  0.56271034
  0.55348146 0.5492375  0.54465336 0.5341085  0.5322538  0.529959
  0.52958804 0.52097064 0.5209465  0.5203551  0.51969826 0.51601183
  0.51106393 0.51009727 0.50940603 0.5084613  0.5080996  0.5071665
  0.50497293]]
[[  8338  25606  16474  14872  29538  10626   6612  47205 308334    194
    6517   6482  10760  49232   3940  41513   5740  29210  16362   6009
   15533  21267  12044   4871  14973]]


In [3]:
# --- 5. Basic logic for word analogies ---
word_A = "king"
word_B = "queen"
word_C = "bucket"

word_A = "król"
word_B = "królowa"
word_C = "mężczyzna"
word_C = "wiadro"


v_A = wv[word_A]
v_B = wv[word_B]
v_C = wv[word_C]

# Vector arithmetic
X_diff = v_B - v_A
v_D_candidate = v_C + X_diff

# Normalize query vector
v_D_candidate = v_D_candidate.astype('float32')
faiss.normalize_L2(v_D_candidate.reshape(1, -1))

# --- 6. Search nearest neighbors ---
k = 25  # Increase k to account for filtered words
scores, indices = index.search(v_D_candidate.reshape(1, -1), k)

# --- 7. Output ---
print(f"Input: {word_C} + ({word_B} - {word_A})")

# Filter out input words (A, B, C) from the results
filtered_results_count = 0
for i in range(k):
    candidate_word = all_words[indices[0][i]]
    if candidate_word not in [word_A, word_B, word_C]:
        print(f"{filtered_results_count+1}: {candidate_word} (score={scores[0][i]:.4f})")
        filtered_results_count += 1
    if filtered_results_count >= 25: # Display top 5 unique results
        break


Input: wiadro + (królowa - król)
1: balia (score=0.6007)
2: kubeł (score=0.5901)
3: zlew (score=0.5732)
4: wiaderko (score=0.5665)
5: dzbanek (score=0.5627)
6: miska (score=0.5535)
7: cebrzyk (score=0.5492)
8: koromysła (score=0.5447)
9: woda (score=0.5341)
10: garnek (score=0.5323)
11: pojemnik (score=0.5300)
12: wrzucać (score=0.5296)
13: konewka (score=0.5210)
14: służąca (score=0.5209)
15: kubełek (score=0.5204)
16: puszka (score=0.5197)
17: tacka (score=0.5160)
18: umywalka (score=0.5111)
19: bielizna (score=0.5101)
20: pralnia (score=0.5094)
21: ścierka (score=0.5085)
22: szalupa (score=0.5081)
23: kosz (score=0.5072)
24: kuchenka (score=0.5050)


In [37]:
print(X[15])
print(X_c[15])

[-7.166600e-02  4.054760e-01 -4.174190e-01  1.251520e-01  5.684670e-01
  1.817690e-01  6.021000e-03 -2.119560e-01 -2.456670e-01 -6.570032e+00
  5.785600e-02 -7.128120e-01 -1.060160e-01 -1.844040e-01 -9.019960e-01
 -9.663020e-01 -6.327800e-02 -6.078010e-01 -5.375060e-01  4.028160e-01
  9.200880e-01 -5.004680e-01 -5.268480e-01 -2.082000e-02 -6.638070e-01
  6.940480e-01 -3.253220e-01  4.476340e-01  2.509200e-02 -2.883560e-01
 -2.316430e-01 -2.660200e-02  2.022763e+00  6.807710e-01  2.772500e-01
  3.553180e-01 -1.023090e-01  1.528080e-01 -3.501880e-01  9.433600e-02
  1.054920e-01  9.838270e-01  1.791540e-01  1.398500e-02  4.235970e-01
  8.027440e-01 -1.652940e-01 -2.645660e-01  2.717770e-01  1.806170e-01
 -1.804800e-02 -1.419030e-01  6.304230e-01  8.722000e-02 -1.310630e-01
  1.154435e+00 -2.136260e-01 -2.497420e-01  1.199190e+00 -4.389930e-01
 -5.047070e-01 -3.229990e-01  1.578750e-01  2.517610e-01 -1.513634e+00
 -1.297200e-02 -5.682330e-01  6.014700e-02 -2.750540e-01 -2.758580e-01
  3.55

In [3]:
# Probably eighen calculation and space normalisation
import numpy as np
from scipy.linalg import eigh
import time

print("Starting vector processing...")
# ── 1. Load vectors (using pre-loaded wv) ───────────────────────────────────────────
# wv is already loaded by the previous cell (ZxB9XxVWSF6-)
words = list(wv.key_to_index.keys())

X = np.array([wv[w] for w in words]).astype(np.float32)
print(f"Using pre-loaded word vectors (wv) from Gensim. Initial word vectors (X) shape: {X.shape}")

# ── 2. Centre ─────────────────────────────────────────────────
print("Starting centering process...")
t0 = time.time()
mu  = X.mean(axis=0)                  # (dim,)
X_c = X - mu                           # (N, dim) — broadcast
print(f'Centering complete: {time.time()-t0:.2f}s, Centered matrix (X_c) shape: {X_c.shape}')

# ── 3. Covariance  (dim×dim, not N×N) ─────────────────────────
print("Starting covariance calculation...")
t0 = time.time()
cov = (X_c.T @ X_c) / len(X_c)        # (dim, dim)
print(f'Covariance complete: {time.time()-t0:.2f}s, Covariance matrix (cov) shape: {cov.shape}')

# ── 4. Eigendecomposition ─────────────────────────────────────
print("Starting eigendecomposition...")
t0 = time.time()
eigvals, U = eigh(cov)                 # eigvals (dim,), U (dim,dim)
eigvals = eigvals[::-1]               # flip to descending
U       = U[:, ::-1]
eigvals = np.maximum(eigvals, 1e-9)   # regularise near-zeros
print(f'Eigendecomposition complete: {time.time()-t0:.2f}s, Eigenvalues shape: {eigvals.shape}, Eigenvectors (U) shape: {U.shape}')

# ── 5. Build whitening matrix  W = Λ^{-½} Uᵀ ─────────────────
print("Building whitening matrix...")
W = U / np.sqrt(eigvals)               # (dim, dim), cols scaled
W = W.T                                # W shape (dim, dim)
print(f'Whitening matrix (W) built. Shape: {W.shape}')

# ── 6. Transform all vectors  x̃ = W(x−μ) ─────────────────────
print("Transforming all vectors (whitening)...")
t0 = time.time()
X_white = X_c @ W.T                   # (N, dim) — the bottleneck
print(f'Vector transformation complete: {time.time()-t0:.2f}s, Whitened vectors (X_white) shape: {X_white.shape}')

# ── 7. Save ───────────────────────────────────────────────────
print("Saving whitened vectors, whitening matrix, and mean...")
np.save('output-data/X_white.npy',  X_white)      # fast binary
np.save('output-data/W_matrix.npy', W)            # reuse for new words
np.save('output-data/mu.npy',       mu)
with open('output-data/words.txt', 'w', encoding='utf-8') as f:
    f.write('\n'.join(words))
print("Save complete.")

# ── 8. Query new word at runtime ──────────────────────────────
print("Preparing for runtime queries...")
def whiten_vec(x_raw):
    # Ensure x_raw is a numpy array
    x_raw_np = np.asarray(x_raw, dtype=np.float32)
    return W @ (x_raw_np - mu)            # single O(d²) multiply

def cosine(a, b):
    a = a / np.linalg.norm(a)
    b = b / np.linalg.norm(b)
    return float(a @ b)


# example below has no practical meaning, just a test of the new space
word2idx = {w: i for i, w in enumerate(words)}
# Example usage:
# Check if words 'bank', 'river', 'loan' exist in the vocabulary before querying
test_words = ['bank', 'river', 'loan'] # These are English words, assuming Polish model has them or equivalent
available_test_words = [w for w in test_words if w in word2idx]

if len(available_test_words) >= 3:
    bank_vec  = X_white[word2idx['zamek']]
    river_vec = X_white[word2idx['rzeka']]
    loan_vec  = X_white[word2idx['pożyczka']]

    print(f"Cosine similarity between 'bank' and 'river': {cosine(bank_vec, river_vec)}")
    print(f"Cosine similarity between 'bank' and 'loan': {cosine(bank_vec, loan_vec)}")
else:
    print(f"Not all test words ({test_words}) found in vocabulary for similarity calculation. Available words: {list(word2idx.keys())[:10]}...")

print("Cell S7mm7qcZYOTz execution finished.")


Starting vector processing...
Using pre-loaded word vectors (wv) from Gensim. Initial word vectors (X) shape: (1926320, 100)
Starting centering process...
Centering complete: 0.27s, Centered matrix (X_c) shape: (1926320, 100)
Starting covariance calculation...
Covariance complete: 0.37s, Covariance matrix (cov) shape: (100, 100)
Starting eigendecomposition...
Eigendecomposition complete: 0.02s, Eigenvalues shape: (100,), Eigenvectors (U) shape: (100, 100)
Building whitening matrix...
Whitening matrix (W) built. Shape: (100, 100)
Transforming all vectors (whitening)...
Vector transformation complete: 0.18s, Whitened vectors (X_white) shape: (1926320, 100)
Saving whitened vectors, whitening matrix, and mean...
Save complete.
Preparing for runtime queries...
Cosine similarity between 'bank' and 'river': 0.30531415343284607
Cosine similarity between 'bank' and 'loan': 0.23861826956272125
Cell S7mm7qcZYOTz execution finished.


In [39]:
# --- 5. Analogy Logic in Whitened Space   ver 1 --- to be verified if this works
word_A = "król"
word_B = "królowa"
word_C = "mężczyzna"
word_C = "wiadro"

# Get index for the words
idx_A = word2idx[word_A]
idx_B = word2idx[word_B]
idx_C = word2idx[word_C]

# Use pre-computed whitened vectors from X_white
v_A_white = X_white[idx_A]
v_B_white = X_white[idx_B]
v_C_white = X_white[idx_C]

# Vector arithmetic in whitened space
X_diff_white = v_B_white - v_A_white
v_D_candidate_white = v_C_white + X_diff_white

# --- 6. Search nearest neighbors in X_white ---
# Normalize candidate for cosine similarity
query_vec = v_D_candidate_white / np.linalg.norm(v_D_candidate_white)

# X_white is already normalized from previous cell execution steps if FAISS logic was used,
# but to be safe and independent of index state, we normalize a copy or ensure normalization:
X_white_norm = X_white / np.linalg.norm(X_white, axis=1, keepdims=True)

# Calculate all cosine similarities
similarities = X_white_norm @ query_vec

# Get top indices
k = 25
top_indices = np.argsort(similarities)[::-1]

# --- 7. Output ---
print(f"Input: {word_C} + ({word_B} - {word_A}) [Whitened Space]")

filtered_results_count = 0
for idx in top_indices:
    candidate_word = words[idx]
    if candidate_word not in [word_A, word_B, word_C]:
        print(f"{filtered_results_count+1}: {candidate_word} (score={similarities[idx]:.4f})")
        filtered_results_count += 1
    if filtered_results_count >= 25:
        break

Input: wiadro + (królowa - król) [Whitened Space]
1: kubeł (score=0.6029)
2: cebrzyk (score=0.5886)
3: zlew (score=0.5564)
4: balia (score=0.5440)
5: dzbanek (score=0.5347)
6: jasicowa (score=0.5322)
7: wiaderko (score=0.5259)
8: kubełek (score=0.5233)
9: pojemnik (score=0.5207)
10: wietliny (score=0.5173)
11: garnek (score=0.5131)
12: kubek (score=0.5085)
13: wydestylowywania (score=0.5085)
14: woda (score=0.5035)
15: koromysła (score=0.4975)
16: bezkaloryczny (score=0.4965)
17: konew (score=0.4956)
18: brachiczna (score=0.4947)
19: wrzątek (score=0.4927)
20: naczynie (score=0.4918)
21: szklanke (score=0.4871)
22: puszka (score=0.4865)
23: ścierka (score=0.4863)
24: beczka (score=0.4854)
25: nalewać (score=0.4849)


# All below to verify

## Advanced FAISS Search (IVF Algorithm)

In [41]:
import faiss
import numpy as np

# --- 5. Analogy Logic in Whitened Space ---
word_A = "król"
word_B = "królowa"
word_C = "wiadro"

idx_A, idx_B, idx_C = word2idx[word_A], word2idx[word_B], word2idx[word_C]

v_A_white = X_white[idx_A]
v_B_white = X_white[idx_B]
v_C_white = X_white[idx_C]

X_diff_white = v_B_white - v_A_white
v_D_candidate_white = v_C_white + X_diff_white

# --- 6. Advanced FAISS Search (IVF Algorithm) ---
print("Configuring FAISS IndexIVFFlat (Partitioned KNN search)...")
d_white = X_white.shape[1]

# 1. Normalization
X_white_norm = X_white.copy().astype('float32')
faiss.normalize_L2(X_white_norm)

# 2. Define the algorithm: IndexIVFFlat (Voronoi partitions)
# nlist is the number of clusters (cells)
nlist = int(np.sqrt(X_white_norm.shape[0]))
quantizer = faiss.IndexFlatIP(d_white) # Coarse quantizer
index_ivf = faiss.IndexIVFFlat(quantizer, d_white, nlist, faiss.METRIC_INNER_PRODUCT)

# 3. Train the index (Required for IVF algorithms)
print(f"Training IVF index on {X_white_norm.shape[0]} vectors with {nlist} clusters...")
index_ivf.train(X_white_norm)
index_ivf.add(X_white_norm)

# 4. Set nprobe (How many neighboring clusters to search)
index_ivf.nprobe = 10

# 5. Prepare query vector
query_vec = v_D_candidate_white.reshape(1, -1).astype('float32')
faiss.normalize_L2(query_vec)

# 6. Execute KNN search algorithm
k = 25
scores, indices = index_ivf.search(query_vec, k)

# --- 7. Output ---
print(f"\nInput: {word_C} + ({word_B} - {word_A}) [Whitened Space via IVF-KNN]")
filtered_results_count = 0
for i in range(k):
    idx = indices[0][i]
    if idx == -1: continue # Skip if search failed to find enough neighbors
    candidate_word = words[idx]
    if candidate_word not in [word_A, word_B, word_C]:
        print(f"{filtered_results_count+1}: {candidate_word} (score={scores[0][i]:.4f})")
        filtered_results_count += 1
    if filtered_results_count >= 25:
        break

Configuring FAISS IndexIVFFlat (Partitioned KNN search)...
Training IVF index on 1926320 vectors with 1387 clusters...

Input: wiadro + (królowa - król) [Whitened Space via IVF-KNN]
1: kubeł (score=0.6029)
2: cebrzyk (score=0.5886)
3: zlew (score=0.5564)
4: balia (score=0.5440)
5: dzbanek (score=0.5347)
6: jasicowa (score=0.5322)
7: wiaderko (score=0.5259)
8: kubełek (score=0.5233)
9: pojemnik (score=0.5207)
10: wietliny (score=0.5173)
11: garnek (score=0.5131)
12: kubek (score=0.5085)
13: wydestylowywania (score=0.5085)
14: woda (score=0.5035)
15: koromysła (score=0.4975)
16: bezkaloryczny (score=0.4965)
17: konew (score=0.4956)
18: brachiczna (score=0.4947)
19: wrzątek (score=0.4927)
20: naczynie (score=0.4918)
21: szklanke (score=0.4871)
22: puszka (score=0.4865)
23: ścierka (score=0.4863)
24: beczka (score=0.4854)


In [ ]:
# --- 5. Analogy Logic in Whitened Space   ver 2 --- to be verified if this works
import faiss
import numpy as np

# --- 5. Analogy Logic in Whitened Space ---
word_A = "król"
word_B = "królowa"
word_C = "wiadro"

# Retrieve indices from word2idx created in the whitening cell
idx_A, idx_B, idx_C = word2idx[word_A], word2idx[word_B], word2idx[word_C]

# Select the whitened vectors
v_A_white = X_white[idx_A]
v_B_white = X_white[idx_B]
v_C_white = X_white[idx_C]

# Compute the target candidate vector (C + B - A)
X_diff_white = v_B_white - v_A_white
v_D_candidate_white = v_C_white + X_diff_white

# --- 6. FAISS K-Nearest Neighbors Search ---
print("Searching for closest neighbors using FAISS...")
d_white = X_white.shape[1]

# We use IndexFlatIP (Inner Product) because it's equivalent to Cosine Similarity
# when the vectors are L2-normalized.
index_white = faiss.IndexFlatIP(d_white)

# Normalize all vectors to unit length
X_white_norm = X_white.copy().astype('float32')
faiss.normalize_L2(X_white_norm)

# Add vectors to the FAISS index
index_white.add(X_white_norm)

# Prepare the query: normalize the candidate vector
query_vec = v_D_candidate_white.reshape(1, -1).astype('float32')
faiss.normalize_L2(query_vec)

# Search for the k closest neighbors
k = 25
scores, indices = index_white.search(query_vec, k)

# --- 7. Results ---
print(f"\nInput analogy: {word_C} + ({word_B} - {word_A})")
print("Closest neighbors (excluding inputs):")

filtered_results_count = 0
for i in range(k):
    idx = indices[0][i]
    candidate_word = words[idx]
    if candidate_word not in [word_A, word_B, word_C]:
        print(f"{filtered_results_count+1}: {candidate_word} (score={scores[0][i]:.4f})")
        filtered_results_count += 1
    if filtered_results_count >= 25:
        break

Searching for closest neighbors using FAISS...

Input analogy: wiadro + (królowa - król)
Closest neighbors (excluding inputs):
1: kubeł (score=0.6029)
2: cebrzyk (score=0.5886)
3: zlew (score=0.5564)
4: balia (score=0.5440)
5: dzbanek (score=0.5347)
6: jasicowa (score=0.5322)
7: wiaderko (score=0.5259)
8: kubełek (score=0.5233)
9: pojemnik (score=0.5207)
10: wietliny (score=0.5173)
11: garnek (score=0.5131)
12: kubek (score=0.5085)
13: wydestylowywania (score=0.5085)
14: woda (score=0.5035)
15: koromysła (score=0.4975)
16: bezkaloryczny (score=0.4965)
17: konew (score=0.4956)
18: brachiczna (score=0.4947)
19: wrzątek (score=0.4927)
20: naczynie (score=0.4918)
21: szklanke (score=0.4871)
22: puszka (score=0.4865)
23: ścierka (score=0.4863)
24: beczka (score=0.4854)


In [50]:
import numpy as np

# --- Parameters ---
# Using a fixed numeric value as requested
margin_value = 6
word_A = "król"
word_B = "królowa"
word_C = "mężczyzna"
word_C = "wiadro"

# --- 1. Prepare candidate ---
# Using variables from the previous whitening/index cells
idx_A, idx_B, idx_C = word2idx[word_A], word2idx[word_B], word2idx[word_C]
v_A_white = X_white[idx_A]
v_B_white = X_white[idx_B]
v_C_white = X_white[idx_C]

# Compute the target candidate vector (C + B - A)
v_D_candidate_white = v_C_white + (v_B_white - v_A_white)

# --- 2. Define the Hypercube Bounds ---
# The cube is +/- 0.01 absolute numeric value per dimension
lower_bounds = v_D_candidate_white - margin_value
upper_bounds = v_D_candidate_white + margin_value

print(f"Searching for words inside a {X_white.shape[1]}D-cube (!{margin_value}) around the candidate...")

# --- 3. Find vectors within bounds ---
# Check which vectors in X_white satisfy lower <= value <= upper for ALL dimensions
mask = np.all((X_white >= lower_bounds) & (X_white <= upper_bounds), axis=1)

# Get indices and compute distances, excluding input words
inside_indices = np.where(mask)[0]
results = []
for i in inside_indices:
    w = words[i]
    if w in [word_A, word_B, word_C]:
        continue
    dist = np.linalg.norm(X_white[i] - v_D_candidate_white)
    results.append((w, i, float(dist)))

# Sort by ascending distance (closest first)
results.sort(key=lambda t: t[2])

# --- 4. Display Results ---
print(f"Total words found inside the hypercube: {len(results)}")
print("--------------------------------------------------")
if len(results) > 0:
    for rank, (word, idx, dist) in enumerate(results, start=1):
        print(f"{rank}: {word} (Euclidean distance to center: {dist:.4f})")
else:
    print(f"No words found within the !{margin_value} range.")

Searching for words inside a 100D-cube (!6) around the candidate...
Total words found inside the hypercube: 44
--------------------------------------------------
1: zlew (Euclidean distance to center: 22.1131)
2: pojemnik (Euclidean distance to center: 22.5804)
3: beczka (Euclidean distance to center: 23.0004)
4: garnek (Euclidean distance to center: 23.1672)
5: nalewać (Euclidean distance to center: 23.1679)
6: zanurzać (Euclidean distance to center: 23.2486)
7: gulbin (Euclidean distance to center: 23.2915)
8: miseczka (Euclidean distance to center: 23.6355)
9: czajnik (Euclidean distance to center: 23.7735)
10: pokojówka (Euclidean distance to center: 23.8850)
11: pluskać (Euclidean distance to center: 23.9910)
12: betsy (Euclidean distance to center: 24.0383)
13: podbiegły (Euclidean distance to center: 24.2104)
14: spuszczać (Euclidean distance to center: 24.3572)
15: brylewska (Euclidean distance to center: 24.4256)
16: brodzik (Euclidean distance to center: 24.7907)
17: żaglówka

In [ ]:
import numpy as np

# --- Parameters ---
# Using a fixed numeric value as requested
margin_value = 6
word_A = "król"
word_B = "królowa"
word_C = "mężczyzna"
word_C = "wiadro"

# --- 1. Prepare candidate ---
# Using variables from the previous whitening/index cells
idx_A, idx_B, idx_C = word2idx[word_A], word2idx[word_B], word2idx[word_C]
v_A_white = X_white[idx_A]
v_B_white = X_white[idx_B]
v_C_white = X_white[idx_C]

# Compute the target candidate vector (C + B - A)
v_D_candidate_white = v_C_white + (v_B_white - v_A_white)

# --- 2. Define the Hypercube Bounds ---
# The cube is +/- 0.01 absolute numeric value per dimension
lower_bounds = v_D_candidate_white - margin_value
upper_bounds = v_D_candidate_white + margin_value

print(f"Searching for words inside a {X_white.shape[1]}D-cube (!{margin_value}) around the candidate...")

# --- 3. Find vectors within bounds ---
# Check which vectors in X_white satisfy lower <= value <= upper for ALL dimensions
mask = np.all((X_white >= lower_bounds) & (X_white <= upper_bounds), axis=1)

# Get indices and words
inside_indices = np.where(mask)[0]
results = [(words[i], i) for i in inside_indices if words[i] not in [word_A, word_B, word_C]]

# --- 4. Display Results ---
print(f"Total words found inside the hypercube: {len(results)}")
print("--------------------------------------------------")
for i, (word, idx) in enumerate(results):
    # Calculate Euclidean distance for reference
    dist = np.linalg.norm(X_white[idx] - v_D_candidate_white)
    print(f"{i+1}: {word} (Euclidean distance to center: {dist:.4f})")

if len(results) == 0:
    print(f"No words found within the !{margin_value} range.")

In [ ]:
import numpy as np

# Diagnostic Cell: Check data alignment and range
print(f"Shape of X_white: {X_white.shape}")
print(f"Number of words in word2idx: {len(word2idx)}")

# Check a specific word's whitened vector
try:
    test_word = 'król'
    idx = word2idx[test_word]
    vec = X_white[idx]
    print(f"\nWhitened vector for '{test_word}' (first 5 dims): {vec[:5]}")
    print(f"Mean of this vector: {np.mean(vec):.4f}")
    print(f"Std Dev of this vector: {np.std(vec):.4f}")

    # Check the range of the candidate vector
    v_D = v_C_white + (v_B_white - v_A_white)
    print(f"\nCandidate D (first 5 dims): {v_D[:5]}")

    # Check min/max across all vectors to understand the space scale
    print(f"Global Min in X_white: {np.min(X_white):.4f}")
    print(f"Global Max in X_white: {np.max(X_white):.4f}")

    # Debug the mask for a large margin
    temp_margin = 1.0
    low, high = v_D - temp_margin, v_D + temp_margin
    mask_test = np.all((X_white >= low) & (X_white <= high), axis=1)
    print(f"\nWords found with margin 1.0: {np.sum(mask_test)}")

except Exception as e:
    print(f"Error during diagnostics: {e}")

Shape of X_white: (1926320, 100)
Number of words in word2idx: 1926320

Whitened vector for 'król' (first 5 dims): [-7.2682567  -0.25737187 -2.8894212   1.7560194   3.7755213 ]
Mean of this vector: 0.0247
Std Dev of this vector: 2.5325

Candidate D (first 5 dims): [-6.3898926  -0.5971104   1.365499   -1.2411691  -0.16142559]
Global Min in X_white: -14.5455
Global Max in X_white: 16.4090

Words found with margin 1.0: 0


In [53]:
# Per-dimension min/max: whitened and original
import numpy as np
try:
    min_per_dim_white = np.min(X_white, axis=0)
    max_per_dim_white = np.max(X_white, axis=0)
    min_per_dim_X = np.min(X, axis=0)
    max_per_dim_X = np.max(X, axis=0)
    for dim in range(min_per_dim_white.shape[0]):
        print(f"{dim+1}) white min={min_per_dim_white[dim]:.6f} ; white max={max_per_dim_white[dim]:.6f} ; X min={min_per_dim_X[dim]:.6f} ; X max={max_per_dim_X[dim]:.6f}")
except Exception as e:
    print('Error computing per-dim stats:', e)

1) white min=-14.545494 ; white max=3.868325 ; X min=-2.496925 ; X max=2.511439
2) white min=-8.147099 ; white max=6.660201 ; X min=-2.290737 ; X max=2.533716
3) white min=-6.640708 ; white max=9.661278 ; X min=-2.493934 ; X max=2.255976
4) white min=-6.183819 ; white max=10.692253 ; X min=-2.230011 ; X max=2.407332
5) white min=-8.764975 ; white max=8.530283 ; X min=-2.452233 ; X max=2.609885
6) white min=-8.089653 ; white max=9.077292 ; X min=-2.276325 ; X max=2.459650
7) white min=-9.176516 ; white max=6.538993 ; X min=-2.608315 ; X max=2.291014
8) white min=-9.304182 ; white max=7.643683 ; X min=-2.310930 ; X max=2.956004
9) white min=-10.815841 ; white max=7.141604 ; X min=-2.358472 ; X max=2.248979
10) white min=-8.364252 ; white max=6.566706 ; X min=-7.273584 ; X max=1.724311
11) white min=-13.177452 ; white max=7.945207 ; X min=-2.956140 ; X max=2.252711
12) white min=-10.139920 ; white max=8.006129 ; X min=-2.483591 ; X max=2.325992
13) white min=-6.796077 ; white max=10.30243

## Sigma (σ) scaling — margin = number of standard deviations

In [59]:
import numpy as np

# --- Parameters ---
# Margin expressed as number of standard deviations (σ).
# After PCA-whitening each dimension has Var=1, so σ_per_dim ≈ 1.
# This makes the margin explicit and self-documenting.
margin_sigma = 6.0
word_A = "król"
word_B = "królowa"
word_C = "wiadro"

# --- 1. Prep ---
idx_A, idx_B, idx_C = word2idx[word_A], word2idx[word_B], word2idx[word_C]
v_A_white = X_white[idx_A]
v_B_white = X_white[idx_B]
v_C_white = X_white[idx_C]
v_D_candidate_white = v_C_white + (v_B_white - v_A_white)

# --- 2. Per-dimension σ (should be ≈1 everywhere after whitening) ---
sigma_per_dim = np.std(X_white, axis=0)          # shape (dim,)
print(f"σ range across dims: {sigma_per_dim.min():.4f} – {sigma_per_dim.max():.4f}  (expected ≈1)")

# --- 3. Hypercube bounds ---
lower_bounds = v_D_candidate_white - margin_sigma * sigma_per_dim
upper_bounds = v_D_candidate_white + margin_sigma * sigma_per_dim

print(f"Searching inside a {X_white.shape[1]}D-cube (±{margin_sigma}σ per dimension)...")

# --- 4. Mask + distances ---
mask = np.all((X_white >= lower_bounds) & (X_white <= upper_bounds), axis=1)
inside_indices = np.where(mask)[0]
results = []
for i in inside_indices:
    w = words[i]
    if w in [word_A, word_B, word_C]:
        continue
    dist = np.linalg.norm(X_white[i] - v_D_candidate_white)
    results.append((w, i, float(dist)))
results.sort(key=lambda t: t[2])

# --- 5. Display ---
print(f"Total words found: {len(results)}")
print("--------------------------------------------------")
if results:
    for rank, (word, idx, dist) in enumerate(results, start=1):
        print(f"{rank}: {word} (Euclidean dist: {dist:.4f})")
else:
    print(f"No words found within ±{margin_sigma}σ range.")

σ range across dims: 0.9983 – 0.9989  (expected ≈1)
Searching inside a 100D-cube (±6.0σ per dimension)...
Total words found: 43
--------------------------------------------------
1: zlew (Euclidean dist: 22.1131)
2: pojemnik (Euclidean dist: 22.5804)
3: beczka (Euclidean dist: 23.0004)
4: garnek (Euclidean dist: 23.1672)
5: nalewać (Euclidean dist: 23.1679)
6: zanurzać (Euclidean dist: 23.2486)
7: gulbin (Euclidean dist: 23.2915)
8: miseczka (Euclidean dist: 23.6355)
9: czajnik (Euclidean dist: 23.7735)
10: pokojówka (Euclidean dist: 23.8850)
11: pluskać (Euclidean dist: 23.9910)
12: betsy (Euclidean dist: 24.0383)
13: podbiegły (Euclidean dist: 24.2104)
14: spuszczać (Euclidean dist: 24.3572)
15: brylewska (Euclidean dist: 24.4256)
16: brodzik (Euclidean dist: 24.7907)
17: żaglówka (Euclidean dist: 24.8644)
18: motorówka (Euclidean dist: 24.8901)
19: brukalskiej (Euclidean dist: 24.9760)
20: parapet (Euclidean dist: 25.1523)
21: strużka (Euclidean dist: 25.2188)
22: kropelka (Euclidea

##  IQR scaling — margin = number of inter-quartile ranges

In [63]:
import numpy as np

# --- Parameters ---
# Margin expressed in IQR units.
# IQR = Q75 - Q25.  For a normal dist: IQR ≈ 1.349σ → 1.5 IQR ≈ 2.0σ.
# Robust to outliers because it uses quantiles, not variance.
margin_iqr = 6
word_A = "król"
word_B = "królowa"
word_C = "wiadro"

# --- 1. Prep ---
idx_A, idx_B, idx_C = word2idx[word_A], word2idx[word_B], word2idx[word_C]
v_A_white = X_white[idx_A]
v_B_white = X_white[idx_B]
v_C_white = X_white[idx_C]
v_D_candidate_white = v_C_white + (v_B_white - v_A_white)

# --- 2. Per-dimension IQR ---
q25 = np.percentile(X_white, 25, axis=0)         # shape (dim,)
q75 = np.percentile(X_white, 75, axis=0)
iqr_per_dim = q75 - q25
# Approximate σ equivalent for reference
sigma_equiv = iqr_per_dim / 1.349
print(f"IQR range across dims: {iqr_per_dim.min():.4f} – {iqr_per_dim.max():.4f}")
print(f"≈σ equivalent:         {sigma_equiv.min():.4f} – {sigma_equiv.max():.4f}")

# --- 3. Hypercube bounds ---
lower_bounds = v_D_candidate_white - margin_iqr * iqr_per_dim
upper_bounds = v_D_candidate_white + margin_iqr * iqr_per_dim

print(f"Searching inside a {X_white.shape[1]}D-cube (±{margin_iqr} IQR per dimension)...")

# --- 4. Mask + distances ---
mask = np.all((X_white >= lower_bounds) & (X_white <= upper_bounds), axis=1)
inside_indices = np.where(mask)[0]
results = []
for i in inside_indices:
    w = words[i]
    if w in [word_A, word_B, word_C]:
        continue
    dist = np.linalg.norm(X_white[i] - v_D_candidate_white)
    results.append((w, i, float(dist)))
results.sort(key=lambda t: t[2])

# --- 5. Display ---
print(f"Total words found: {len(results)}")
print("--------------------------------------------------")
if results:
    for rank, (word, idx, dist) in enumerate(results, start=1):
        print(f"{rank}: {word} (Euclidean dist: {dist:.4f})")
else:
    print(f"No words found within ±{margin_iqr} IQR range.")

IQR range across dims: 0.8790 – 1.1489
≈σ equivalent:         0.6516 – 0.8517
Searching inside a 100D-cube (±6 IQR per dimension)...
Total words found: 96
--------------------------------------------------
1: zlew (Euclidean dist: 22.1131)
2: pojemnik (Euclidean dist: 22.5804)
3: kuchenny (Euclidean dist: 22.8530)
4: szalupa (Euclidean dist: 22.9786)
5: beczka (Euclidean dist: 23.0004)
6: garnek (Euclidean dist: 23.1672)
7: nalewać (Euclidean dist: 23.1679)
8: kubek (Euclidean dist: 23.2354)
9: gulbin (Euclidean dist: 23.2915)
10: cryptonanus (Euclidean dist: 23.7188)
11: czajnik (Euclidean dist: 23.7735)
12: pokojówka (Euclidean dist: 23.8850)
13: pluskać (Euclidean dist: 23.9910)
14: betsy (Euclidean dist: 24.0383)
15: cassat (Euclidean dist: 24.1619)
16: koryfena (Euclidean dist: 24.1823)
17: podbiegły (Euclidean dist: 24.2104)
18: pelagialnych (Euclidean dist: 24.2906)
19: branwi (Euclidean dist: 24.3309)
20: przyglšdał (Euclidean dist: 24.4075)
21: brylewska (Euclidean dist: 24.42

## Raw percent-of-range scaling — margin = fraction of each dimension's full span

In [ ]:
import numpy as np

# --- Parameters ---
# Margin expressed as a fraction of each dimension's (max - min) range.
# E.g. 0.05 = ±5% of the full per-dimension span.
# Simple but sensitive to outliers — prefer σ/IQR when outliers exist.
margin_pct = 0.05
word_A = "król"
word_B = "królowa"
word_C = "wiadro"

# --- 1. Prep ---
idx_A, idx_B, idx_C = word2idx[word_A], word2idx[word_B], word2idx[word_C]
v_A_white = X_white[idx_A]
v_B_white = X_white[idx_B]
v_C_white = X_white[idx_C]
v_D_candidate_white = v_C_white + (v_B_white - v_A_white)

# --- 2. Per-dimension range ---
dim_min  = np.min(X_white, axis=0)               # shape (dim,)
dim_max  = np.max(X_white, axis=0)
dim_range = dim_max - dim_min
margin_abs = margin_pct * dim_range              # absolute margin per dim
print(f"Range (max-min) across dims: {dim_range.min():.4f} – {dim_range.max():.4f}")
print(f"Absolute margin ({margin_pct*100:.1f}%):  {margin_abs.min():.4f} – {margin_abs.max():.4f}")

# --- 3. Hypercube bounds ---
lower_bounds = v_D_candidate_white - margin_abs
upper_bounds = v_D_candidate_white + margin_abs

print(f"Searching inside a {X_white.shape[1]}D-cube (±{margin_pct*100:.1f}% of range per dimension)...")

# --- 4. Mask + distances ---
mask = np.all((X_white >= lower_bounds) & (X_white <= upper_bounds), axis=1)
inside_indices = np.where(mask)[0]
results = []
for i in inside_indices:
    w = words[i]
    if w in [word_A, word_B, word_C]:
        continue
    dist = np.linalg.norm(X_white[i] - v_D_candidate_white)
    results.append((w, i, float(dist)))
results.sort(key=lambda t: t[2])

# --- 5. Display ---
print(f"Total words found: {len(results)}")
print("--------------------------------------------------")
if results:
    for rank, (word, idx, dist) in enumerate(results, start=1):
        print(f"{rank}: {word} (Euclidean dist: {dist:.4f})")
else:
    print(f"No words found within ±{margin_pct*100:.1f}% range.")

## Ellipsoidal (Mahalanobis) search — Sigma variant

In [12]:
# word2idx["wpisz"],
words.index("wpisz")

ValueError: 'wpisz' is not in list

In [8]:
import numpy as np

# --- Parameters ---
# Ellipsoidal search: Mahalanobis distance using per-dimension σ.
# d_M(x) = sqrt( Σ_d ((x_d - c_d) / σ_d)² )
#
# WHY chi-squared quantile was WRONG:
#   χ²(d) describes ||x||² when x~N(0,I)  (distance from ORIGIN).
#   The candidate c = v_C + v_B - v_A is NOT at the origin.
#   Real dist²  ~ non-central χ²(d, λ=||c||²),  E[dist] = sqrt(d + ||c||²).
#   Fix: compute all distances empirically and use the coverage percentile directly.
coverage_pct = 0.0010   # top fraction of all words to include (0.10 = closest 10%)
word_A = "król"
word_B = "królowa"
word_C = "wiadro"

# "tu wpisz swoj tekst piosenki tutaj"

word_A = "tu"
word_B = "wpisz"
word_C = "wiadro"

# --- 1. Prep ---
idx_A, idx_B, idx_C = word2idx[word_A], word2idx[word_B], word2idx[word_C]
v_A_white = X_white[idx_A]
v_B_white = X_white[idx_B]
v_C_white = X_white[idx_C]
v_D_candidate_white = v_C_white + (v_B_white - v_A_white)

# --- 2. Per-dimension σ + EMPIRICAL radius from coverage percentile ---
sigma_per_dim = np.std(X_white, axis=0)           # (dim,) — ≈1 after whitening
d = X_white.shape[1]
diff = X_white - v_D_candidate_white              # (N, dim)  broadcast
mahal_dist = np.sqrt(np.sum((diff / sigma_per_dim) ** 2, axis=1))  # (N,)

radius_sigma = float(np.percentile(mahal_dist, coverage_pct * 100))
print(f"d={d}, ||candidate||={np.linalg.norm(v_D_candidate_white):.2f}")
print(f"Empirical dist: min={mahal_dist.min():.2f}  median={np.median(mahal_dist):.2f}  max={mahal_dist.max():.2f}")
print(f"Coverage {coverage_pct:.0%}  →  empirical radius = {radius_sigma:.3f}σ")
print(f"σ range across dims: {sigma_per_dim.min():.4f} – {sigma_per_dim.max():.4f}  (expected ≈1 after whitening)")

# --- 3. Filter by radius ---
mask = mahal_dist <= radius_sigma
inside_indices = np.where(mask)[0]
results = []
for i in inside_indices:
    w = words[i]
    if w in [word_A, word_B, word_C]:
        continue
    results.append((w, i, float(mahal_dist[i])))
results.sort(key=lambda t: t[2])

# --- 4. Display ---
print(f"Ellipsoidal search (radius ≤ {radius_sigma:.3f}σ, coverage ≈{coverage_pct:.0%}): {len(results)} words found")
print("--------------------------------------------------")
if results:
    for rank, (word, idx, dist) in enumerate(results, start=1):
        print(f"{rank}: {word} (Mahalanobis dist: {dist:.4f}σ)")
else:
    print(f"No words found within {radius_sigma:.3f}σ radius.")

KeyError: 'wpisz'

## Song transfer using sigma Mahalanobis diffs (adaptive coverage)

In [5]:
import numpy as np
import re
import unicodedata

# Build a tokenizer that keeps Polish letters and apostrophes.
def _tokenize_song(text):
    return re.findall(r"[^\W\d_]+(?:'[^\W\d_]+)?", text.lower(), flags=re.UNICODE)


def _ascii_key(token):
    # Fold diacritics: swój -> swoj
    return "".join(
        ch for ch in unicodedata.normalize("NFKD", token.lower())
        if not unicodedata.combining(ch)
    )


def _build_ascii_map(vocab_words):
    # Map ASCII-folded token -> preferred vocab word.
    # If many words collapse to same key, keep the shortest as a simple heuristic.
    mapping = {}
    for w in vocab_words:
        k = _ascii_key(w)
        if (k not in mapping) or (len(w) < len(mapping[k])):
            mapping[k] = w
    return mapping


ASCII_MAP = _build_ascii_map(words)


def _resolve_vocab_token(token):
    """
    Resolve token to vocabulary using:
    1) exact; 2) lowercase; 3) diacritic-folded match.
    """
    if token in word2idx:
        return token, "exact"
    t = token.lower()
    if t in word2idx:
        return t, "lower"
    k = _ascii_key(t)
    if k in ASCII_MAP:
        return ASCII_MAP[k], "ascii_fold"
    return None, "oov"


def _closest_candidate_sigma(target_vec, sigma_per_dim, banned_words=None, initial_coverage=1e-4, growth=10.0, max_coverage=0.5):
    """
    Find nearest candidate around target_vec using sigma-normalized Mahalanobis distance.
    Coverage expands adaptively until at least one valid candidate appears.
    """
    if banned_words is None:
        banned_words = set()

    diff = X_white - target_vec
    dist = np.sqrt(np.sum((diff / sigma_per_dim) ** 2, axis=1))

    coverage = float(initial_coverage)

    while coverage <= max_coverage:
        radius = float(np.percentile(dist, coverage * 100.0))
        candidate_idx = np.where(dist <= radius)[0]

        if candidate_idx.size > 0:
            ordered = candidate_idx[np.argsort(dist[candidate_idx])]
            for idx in ordered:
                w = words[idx]
                if w not in banned_words:
                    idx = int(idx)
                    return idx, float(dist[idx]), coverage, radius

        coverage *= growth

    # Fallback: nearest valid globally.
    ordered_all = np.argsort(dist)
    for idx in ordered_all:
        w = words[idx]
        if w not in banned_words:
            idx = int(idx)
            return idx, float(dist[idx]), max_coverage, float("nan")

    return None, float("nan"), max_coverage, float("nan")


def generate_song_by_diffs_sigma(song_text, new_start_word, initial_coverage=1e-4, growth=10.0, max_coverage=0.5):
    """
    Algorithm:
    1) For each source pair A=word_n, B=word_{n+1}, compute diff = v_B_white - v_A_white.
    2) Apply diff to current generated word vector: target = v_current + diff.
    3) Find nearest candidate with sigma Mahalanobis, expanding coverage if needed.
    4) Chosen word becomes next current word.

    OOV handling policy (always returns; avoids degenerate exact jumps):
    - Resolve source tokens via exact/lower/ascii-fold.
    - If unresolved, fallback to PREVIOUS RESOLVED SOURCE token (not current generated word).
    - This keeps diffs in the source-trajectory frame and prevents artifacts like
      target becoming exactly B when A is unresolved.
    """
    tokens = _tokenize_song(song_text)
    if len(tokens) < 2:
        raise ValueError("song_text must contain at least 2 words.")

    start_resolved, _ = _resolve_vocab_token(new_start_word)
    if start_resolved is None:
        raise ValueError(f"new_start_word '{new_start_word}' is not in vocabulary (even after ascii-fold).")

    sigma_per_dim = np.std(X_white, axis=0)
    sigma_per_dim = np.where(sigma_per_dim <= 1e-12, 1.0, sigma_per_dim)

    generated = [start_resolved]
    current_word = start_resolved
    debug_rows = []

    # Source-side anchor: last resolved source token.
    first_src_resolved, first_mode = _resolve_vocab_token(tokens[0])
    if first_src_resolved is None:
        first_src_resolved = start_resolved
        first_mode = "fallback_start"
    prev_source_resolved = first_src_resolved

    for i in range(len(tokens) - 1):
        src_A = tokens[i]
        src_B = tokens[i + 1]

        A_resolved, A_mode = _resolve_vocab_token(src_A)
        B_resolved, B_mode = _resolve_vocab_token(src_B)

        used_fallback = False

        if A_resolved is None:
            A_resolved = prev_source_resolved
            A_mode = "fallback_prev_source"
            used_fallback = True

        if B_resolved is None:
            # If B is OOV, stay near source continuity.
            B_resolved = A_resolved
            B_mode = "fallback_A_source"
            used_fallback = True

        idxA = word2idx[A_resolved]
        idxB = word2idx[B_resolved]
        idxCur = word2idx[current_word]

        diff_vec = X_white[idxB] - X_white[idxA]
        target_vec = X_white[idxCur] + diff_vec

        banned = {current_word}
        # Strict mode: when the source pair is fully resolved (status=ok),
        # never allow exact copy of B_resolved as the chosen output token.
        if not used_fallback:
            banned.add(B_resolved)
        idx_new, d_new, cov_used, radius_used = _closest_candidate_sigma(
            target_vec,
            sigma_per_dim,
            banned_words=banned,
            initial_coverage=initial_coverage,
            growth=growth,
            max_coverage=max_coverage,
        )

        if idx_new is None:
            generated.append(current_word)
            debug_rows.append({
                "step": i + 1,
                "A": src_A,
                "B": src_B,
                "A_resolved": A_resolved,
                "B_resolved": B_resolved,
                "A_mode": A_mode,
                "B_mode": B_mode,
                "status": "fallback_no_candidate",
                "chosen": current_word,
            })
            prev_source_resolved = B_resolved
            continue

        current_word = words[idx_new]
        generated.append(current_word)

        debug_rows.append({
            "step": i + 1,
            "A": src_A,
            "B": src_B,
            "A_resolved": A_resolved,
            "B_resolved": B_resolved,
            "A_mode": A_mode,
            "B_mode": B_mode,
            "status": "approx_oov_pair" if used_fallback else "ok",
            "chosen": current_word,
            "dist": d_new,
            "coverage_used": cov_used,
            "radius_used": radius_used,
        })

        prev_source_resolved = B_resolved

    new_song_text = " ".join(generated)
    return new_song_text, debug_rows

In [6]:
# --- Usage ---
# Provide your source lyrics and a new starting word from vocabulary.
# Example below uses plain-ASCII Polish on purpose; resolver will try diacritic-fold mapping.
song_text = "tu wpisz swoj tekst piosenki tutaj"
new_start_word = "wiadro"

new_song_text, steps = generate_song_by_diffs_sigma(
    song_text=song_text,
    new_start_word=new_start_word,
    initial_coverage=1e-4,  # starts very tight
    growth=10.0,            # expands: 1e-4 -> 1e-3 -> 1e-2 -> ...
    max_coverage=0.5,       # safety cap before global nearest fallback
)

print("Generated target text:")
print("=" * 80)
print(new_song_text)
print("=" * 80)
print(f"Generated words: {len(new_song_text.split())}")

print("\nFirst steps preview:")
for row in steps[:10]:
    print(row)

Generated target text:
wiadro kubeł swoj autor napisać tutaj
Generated words: 6

First steps preview:
{'step': 1, 'A': 'tu', 'B': 'wpisz', 'A_resolved': 'tu', 'B_resolved': 'tu', 'A_mode': 'exact', 'B_mode': 'fallback_A_source', 'status': 'approx_oov_pair', 'chosen': 'kubeł', 'dist': 11.954596519470215, 'coverage_used': 0.0001, 'radius_used': 18.061155319213867}
{'step': 2, 'A': 'wpisz', 'B': 'swoj', 'A_resolved': 'tu', 'B_resolved': 'swoj', 'A_mode': 'fallback_prev_source', 'B_mode': 'exact', 'status': 'approx_oov_pair', 'chosen': 'swoj', 'dist': 23.971113204956055, 'coverage_used': 0.0001, 'radius_used': 25.70049285888672}
{'step': 3, 'A': 'swoj', 'B': 'tekst', 'A_resolved': 'swoj', 'B_resolved': 'tekst', 'A_mode': 'exact', 'B_mode': 'exact', 'status': 'ok', 'chosen': 'autor', 'dist': 13.95423412322998, 'coverage_used': 0.0001, 'radius_used': 20.062578201293945}
{'step': 4, 'A': 'tekst', 'B': 'piosenki', 'A_resolved': 'tekst', 'B_resolved': 'tekst', 'A_mode': 'exact', 'B_mode': 'fall

## Ellipsoidal (Mahalanobis) search — Robust IQR variant

In [ ]:
import numpy as np

# --- Parameters ---
# Robust ellipsoidal search: Mahalanobis distance using per-dimension IQR.
# d_IQR(x) = sqrt( Σ_d ((x_d - c_d) / IQR_d)² )
#
# WHY chi-squared quantile was WRONG:
#   Same non-centrality issue as the sigma variant.
#   Fix: compute all IQR-distances empirically and use the coverage percentile.
coverage_pct = 0.10   # top fraction of all words to include (0.10 = closest 10%)
word_A = "król"
word_B = "królowa"
word_C = "wiadro"

# --- 1. Prep ---
idx_A, idx_B, idx_C = word2idx[word_A], word2idx[word_B], word2idx[word_C]
v_A_white = X_white[idx_A]
v_B_white = X_white[idx_B]
v_C_white = X_white[idx_C]
v_D_candidate_white = v_C_white + (v_B_white - v_A_white)

# --- 2. Per-dimension IQR + EMPIRICAL radius from coverage percentile ---
q25 = np.percentile(X_white, 25, axis=0)          # (dim,)
q75 = np.percentile(X_white, 75, axis=0)
iqr_per_dim = q75 - q25
sigma_equiv = iqr_per_dim / 1.349
d = X_white.shape[1]
diff = X_white - v_D_candidate_white               # (N, dim)  broadcast
iqr_dist = np.sqrt(np.sum((diff / iqr_per_dim) ** 2, axis=1))  # (N,)

radius_iqr = float(np.percentile(iqr_dist, coverage_pct * 100))
print(f"IQR range across dims: {iqr_per_dim.min():.4f} – {iqr_per_dim.max():.4f}")
print(f"≈σ equivalent:         {sigma_equiv.min():.4f} – {sigma_equiv.max():.4f}")
print(f"d={d}, ||candidate||={np.linalg.norm(v_D_candidate_white):.2f}")
print(f"Empirical IQR-dist: min={iqr_dist.min():.2f}  median={np.median(iqr_dist):.2f}  max={iqr_dist.max():.2f}")
print(f"Coverage {coverage_pct:.0%}  →  empirical radius = {radius_iqr:.3f} IQR-units")

# --- 3. Filter by radius ---
mask = iqr_dist <= radius_iqr
inside_indices = np.where(mask)[0]
results = []
for i in inside_indices:
    w = words[i]
    if w in [word_A, word_B, word_C]:
        continue
    results.append((w, i, float(iqr_dist[i])))
results.sort(key=lambda t: t[2])

# --- 4. Display ---
print(f"Robust ellipsoidal search (radius ≤ {radius_iqr:.3f} IQR, coverage ≈{coverage_pct:.0%}): {len(results)} words found")
print("--------------------------------------------------")
if results:
    for rank, (word, idx, dist) in enumerate(results, start=1):
        print(f"{rank}: {word} (IQR-Mahalanobis dist: {dist:.4f})")
else:
    print(f"No words found within {radius_iqr:.3f} IQR radius.")